In [1]:
# Importation des librairies
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline

In [2]:
# Importation du dataset final (après Data Preparation & Feature Engineering)
df = pd.read_csv("../data/train_data.csv")
df.head()

,Chambres,Superficie_m2,DistanceRoute_m,Quartier,AgeMaison,LoyerMensuel_BIF,Confort_Score,Chambres_par_Superficie
0,4.0,192.0,277.0,Rohero,17.0,2600000,4.0,0.020833
1,5.0,234.0,47.0,Rohero,37.0,2600000,3.0,0.021368
2,3.0,151.0,45.0,Kinanira,23.0,1069218,3.0,0.019868
3,5.0,227.0,174.0,Gasekebuye,22.0,2600000,5.0,0.022026
4,5.0,262.0,216.0,Gihosha,21.0,2085458,3.0,0.019084


In [3]:
# Definir les features(X) et target(y)
X = df.drop(columns=["LoyerMensuel_BIF"])
y = df["LoyerMensuel_BIF"]
y = np.array(y).reshape(-1, 1)

# On applique le log
y = np.log1p(y)

In [4]:
# Train / test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train

,Chambres,Superficie_m2,DistanceRoute_m,Quartier,AgeMaison,Confort_Score,Chambres_par_Superficie
310,3.0,119.0,26.0,Cibitoke,27.0,3.0,0.025210
349,1.0,41.0,187.0,Gasekebuye,34.0,2.0,0.024390
485,5.0,232.0,97.0,Rohero,27.0,5.0,0.021552
137,6.0,284.0,119.0,Kinama,29.0,4.0,0.021127
497,2.0,118.0,90.0,Nyakabiga,34.0,3.0,0.016949
...,...,...,...,...,...,...,...
106,3.0,139.0,43.0,NaN,12.0,2.0,0.021583
270,3.0,124.0,NaN,Gihosha,24.0,4.0,0.024194
348,3.0,NaN,118.0,Musaga,5.0,3.0,NaN
435,4.0,164.0,120.0,Gihosha,3.0,2.0,0.024390


In [5]:
# Creation du preprocessor
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist() 
# "category" pour inclure les catégories pandas

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

In [6]:
# Definir et entrainer le baseline DummyRegressor
dummy_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', DummyRegressor(strategy='mean'))
])
dummy_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [7]:
# Prediction sur le baseline DummyRegressor
y_pred_dummy = dummy_pipeline.predict(X_test)

In [8]:
# Definir et entrainer le baseline LinearRegression
linear_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])
linear_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [9]:
# Prediction sur le baseline LinearRegression
y_pred_linear = linear_pipeline.predict(X_test)

In [10]:
# Métriques et Inverse Log (pour avoir les résultats en BIF réels)
def evaluate_model(true_log, pred_log, model_name):
    # Transformation inverse pour revenir en BIF
    y_true = np.expm1(true_log)
    y_pred = np.expm1(pred_log)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"--- {model_name} ---")
    print(f"MAE (BIF) : {mae:,.0f}")
    print(f"RMSE (BIF) : {rmse:,.0f}")
    print(f"R² (BIF) : {r2:.4f}\n")

    return {'Model': model_name, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

In [11]:
results_dummy = evaluate_model(y_test, y_pred_dummy, "DummyRegressor (Moyenne)")
results_linear = evaluate_model(y_test, y_pred_linear, "Linear Regression")

--- DummyRegressor (Moyenne) ---
MAE (BIF) : 561,706
RMSE (BIF) : 747,383
R² (BIF) : -0.1603

--- Linear Regression ---
MAE (BIF) : 173,325
RMSE (BIF) : 240,864
R² (BIF) : 0.8795

